In [50]:
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import pyvista as pv
import sionna
import tensorflow as tf
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, PathSolver, Camera
from scipy.optimize import minimize
import h5py
from pathlib import Path
import sys

In [51]:
# ============================================================================
# SYSTEM PARAMETERS AND CLASS DEFINITIONS
# ============================================================================

# --- SYSTEM PARAMETERS ---
CARRIER_FREQUENCY = 2.4e9  
BANDWIDTH = 20e6           
NUM_SUBCARRIERS = 64       
SUBCARRIER_SPACING = BANDWIDTH / NUM_SUBCARRIERS
WAVELENGTH = 3e8 / CARRIER_FREQUENCY
NUM_ANTENNAS_RX = 4

# --- Define subcarrier frequencies ---
subcarrier_indices = np.arange(NUM_SUBCARRIERS)
subcarriers = CARRIER_FREQUENCY + (subcarrier_indices - NUM_SUBCARRIERS/2) * SUBCARRIER_SPACING


class CSIAngleEstimator:
    """Angle of Arrival estimator using MUSIC algorithm with planar array support"""
    def __init__(self, num_antennas, wavelength, array_type='planar'):
        self.num_antennas = num_antennas
        self.wavelength = wavelength
        self.antenna_spacing = wavelength / 2
        self.k = 2 * np.pi / wavelength
        self.array_type = array_type
        
        # For 2x2 planar array: define element positions
        if array_type == 'planar' and num_antennas == 4:
            # 2x2 grid positions: (0,0), (1,0), (0,1), (1,1)
            self.pos_x = np.array([0, 1, 0, 1]) * self.antenna_spacing
            self.pos_y = np.array([0, 0, 1, 1]) * self.antenna_spacing
        else:
            # Linear array fallback
            self.pos_x = np.arange(num_antennas) * self.antenna_spacing
            self.pos_y = np.zeros(num_antennas)
    
    def steering_vector(self, azimuth, elevation=0):
        """2D planar array steering vector"""
        # Phase shift based on planar geometry
        phase = self.k * (self.pos_x * np.cos(azimuth) * np.cos(elevation) + 
                          self.pos_y * np.sin(azimuth) * np.cos(elevation))
        return np.exp(1j * phase)
    
    def estimate_aoa_music(self, csi_matrix, num_sources=1, return_spectrum=False):
        # csi_matrix must be [Antennas, Subcarriers]
        # Forward covariance
        R_forward = csi_matrix @ csi_matrix.conj().T / csi_matrix.shape[1]
        
        # Backward covariance (for forward-backward averaging)
        J = np.fliplr(np.eye(self.num_antennas))  # Exchange matrix
        R_backward = J @ R_forward.conj() @ J
        
        # Forward-backward averaged covariance (more robust, better rank)
        R = (R_forward + R_backward) / 2
        
        eigenvalues, eigenvectors = np.linalg.eigh(R)
        idx = eigenvalues.argsort()[::-1]
        noise_subspace = eigenvectors[:, idx[num_sources:]]
        
        # Extended search range: full 360° (-π to π)
        angles = np.linspace(-np.pi, np.pi, 360)
        spectrum = np.zeros(len(angles))
        
        for i, angle in enumerate(angles):
            a = self.steering_vector(angle)
            spectrum[i] = 1.0 / (np.abs(
                a.conj() @ noise_subspace @ noise_subspace.conj().T @ a
            ) + 1e-10)
        
        peak_idx = np.argmax(spectrum)
        aoa_estimate = angles[peak_idx]
        
        if return_spectrum:
            return aoa_estimate, spectrum
        else:
            return aoa_estimate

class FTMRangeEstimator:
    """Range estimator from CSI phase slope (FTM mechanism) with first-arrival detection"""
    def __init__(self, speed_of_light=3e8):
        self.c = speed_of_light
    
    def estimate_range_from_phase(self, csi_subcarriers, subcarrier_spacing):
        phase = np.unwrap(np.angle(csi_subcarriers))
        subcarrier_idx = np.arange(len(phase))
        coeffs = np.polyfit(subcarrier_idx * subcarrier_spacing, phase, 1)
        phase_slope = coeffs[0]
        toa = -phase_slope / (2 * np.pi)
        return abs(self.c * toa)  # One-way delay from CSI, no /2
    
    def estimate_range_from_cir(self, delays, amplitudes, power_threshold_db=-20):
        """
        First-arrival ToA ranging: uses first path with significant power (LoS)
        instead of strongest path which could be NLOS reflection.
        """
        powers = np.abs(amplitudes) ** 2
        max_power = np.max(powers)
        threshold = max_power * (10 ** (power_threshold_db / 10))
        
        # Find first path above threshold (likely LoS)
        valid_mask = powers > threshold
        if np.any(valid_mask):
            valid_delays = delays[valid_mask]
            first_arrival_delay = np.min(valid_delays)
        else:
            # Fallback to minimum delay
            first_arrival_delay = np.min(delays)
        
        return self.c * first_arrival_delay

class BilaterationSolver:
    """Hybrid localization using range and angle (Weighted Least Squares)"""
    @staticmethod
    def hybrid_localization(anchor_positions, ranges, angles, distance_std=0.5, angle_std=0.1):
        # Ensure anchor_positions is a 2D array
        anchor_positions = np.asarray(anchor_positions)
        if anchor_positions.ndim == 1:
            anchor_positions = anchor_positions.reshape(1, -1)
        
        # Ensure ranges and angles are 1D arrays
        ranges = np.asarray(ranges).flatten()
        angles = np.asarray(angles).flatten()
        
        num_anchors = len(anchor_positions)
        distance_weight = 1 / (distance_std**2)
        angle_weight = 1 / (angle_std**2)
        
        def objective(pos):
            # Ensure pos is 1D
            pos = np.asarray(pos).flatten()
            
            # 1. Distance Errors (Ranging)
            error_d = 0.0
            for i in range(num_anchors):
                dist = np.linalg.norm(pos - anchor_positions[i])
                error_d += distance_weight * (ranges[i] - dist)**2
            
            # 2. Angle Errors (AoA)
            error_a = 0.0 
            
            for i in range(num_anchors):
                if not np.isnan(angles[i]):
                    # Extract x,y coordinates as scalars
                    pos_x = float(pos[0])
                    pos_y = float(pos[1])
                    anchor_x = float(anchor_positions[i, 0])
                    anchor_y = float(anchor_positions[i, 1])
                    
                    # Calculate direction FROM ANCHOR TO TARGET
                    dir_x = pos_x - anchor_x
                    dir_y = pos_y - anchor_y
                    
                    estimated_angle = np.arctan2(dir_y, dir_x)
                    angle_diff = float(angles[i]) - estimated_angle
                    
                    # Normalize angle difference to [-pi, pi]
                    angle_diff = np.arctan2(np.sin(angle_diff), np.cos(angle_diff))
                    error_a += angle_weight * (angle_diff**2)
            
            # Total Error
            total_error = error_d + error_a
            return float(total_error)
        
        # Initial guess: mean of anchor positions - ENSURE 1D
        x0 = np.mean(anchor_positions, axis=0).flatten()
        
        # Debug: verify x0 is 1D
        if x0.ndim != 1:
            raise ValueError(f"x0 has {x0.ndim} dimensions, shape {x0.shape}. Expected 1D array.")
        
        result = minimize(objective, x0, method='BFGS') 
        return result.x

# --- IMPAIRMENT AND COMPENSATION FUNCTIONS ---

def add_sfo(csi_matrix, subcarrier_indices, sfo_ppm=5):
    """Adds Sampling Frequency Offset (SFO) to the CSI matrix."""
    clock_mismatch = sfo_ppm * 1e-6
    phase_slope = 2 * np.pi * clock_mismatch
    sfo_phase = phase_slope * subcarrier_indices
    sfo_matrix = np.tile(np.exp(1j * sfo_phase), (csi_matrix.shape[0], 1))
    return csi_matrix * sfo_matrix

def add_phase_noise(csi_matrix, pn_std=0.5):
    """Adds Phase Noise (PN) / Common Phase Error (CPE)."""
    pn_shift = np.random.normal(0, pn_std) 
    pn_factor = np.exp(1j * pn_shift)
    return csi_matrix * pn_factor

def compensate_phase_noise(csi_matrix):
    """Compensates for CPE by removing the mean phase rotation."""
    mean_complex = np.mean(csi_matrix)
    cpe = np.angle(mean_complex)
    return csi_matrix * np.exp(-1j * cpe)

def compensate_sfo(csi_matrix, subcarrier_indices):
    """Compensates for SFO by removing linear phase slope across subcarriers."""
    avg_csi = np.mean(csi_matrix, axis=0)
    avg_phase = np.unwrap(np.angle(avg_csi))
    coeffs = np.polyfit(subcarrier_indices, avg_phase, 1)
    sfo_compensation = np.exp(-1j * np.polyval(coeffs, subcarrier_indices))
    return csi_matrix * sfo_compensation

print("✓ All necessary classes and functions defined.")

✓ All necessary classes and functions defined.


In [52]:
# ============================================================================
# SCENE SETUP AND RAY TRACING
# ============================================================================

no_preview = False

mi.set_variant("llvm_ad_mono_polarized")

scene_path = "../scene/scene_01.xml"

print("Loading scene with Sionna...")
scene = load_scene(scene_path)
print("Sionna scene loaded.")

lambda_half = 0.0625  # half wavelength spacing at 2.4 GHz

# AP as RX (home router, omnidirectional)
scene.rx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="V"
)

# STA as TX (mobile device)
scene.tx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="cross"
)

# --- Transmitter & Receiver Positions ---
tx_positions = [
    np.array([-2.5, 0.5, 0]),
    np.array([3.15, 0.6, -1.1]),
]
rx_positions = [
    np.array([-1.3, 0.65, -2.8]),
]

# --- Add TX/RX to Scene ---
tx_list, rx_list = [], []

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=f"tx_{i}", position=pos, display_radius=0.08)
    scene.add(tx)
    tx_list.append(tx)

for i, pos in enumerate(rx_positions):
    rx = Receiver(name=f"rx_{i}", position=pos, display_radius=0.08)
    scene.add(rx)
    rx_list.append(rx)

# Aim each transmitter toward each receiver
for tx in tx_list:
    for rx in rx_list:
        tx.look_at(rx)

print(f"Placed {len(tx_list)} transmitters and {len(rx_list)} receivers.\n")

camera_pos = [1.5, 1.0, 1.6]
camera_look = [1.5, 1.5, 1.3]

my_cam = Camera(
    position=camera_pos,
    look_at=camera_look,
)

solver_paths = PathSolver()
paths = solver_paths(scene, max_depth=5, los=True, specular_reflection=True, refraction=True)

num_paths = paths.tau.shape[-1]
print(f"Computed {num_paths} propagation paths per TX-RX link.\n")

Loading scene with Sionna...
Sionna scene loaded.
Placed 2 transmitters and 1 receivers.

Computed 53 propagation paths per TX-RX link.



In [53]:
# ============================================================================
# CSI CONVERSION, IMPAIRMENT FLOW, AND DERIVATION
# ============================================================================

# --- Data Extraction from Sionna ---
a, tau = paths.cir(normalize_delays=False, out_type="numpy")
a_snap = a[..., 0] 
tau_snap = tau

# Dynamically determine dimensions from paths object
num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths = a_snap.shape

# Define Estimator/Solver instances
aoa_estimator = CSIAngleEstimator(num_rx_ant, WAVELENGTH)
range_estimator = FTMRangeEstimator()
solver = BilaterationSolver()

# Choose which transmitter to localize (the mobile device)
target_tx_idx = 0  # Change this to localize a different TX

# Storage for derived localization inputs
all_estimated_ranges = []
all_estimated_angles = []
all_anchor_positions = []
true_target_position = tx_list[target_tx_idx].position.numpy().flatten()  # Flatten immediately

print("\n" + "=" * 70)
print(f"Localizing TX-{target_tx_idx} (Mobile Device) using RX measurements (APs)")
print(f"True TX-{target_tx_idx} Position: [{true_target_position[0]:.2f}, {true_target_position[1]:.2f}, {true_target_position[2]:.2f}]")
print("=" * 70)

# Process only links involving the target transmitter
for rx_idx in range(num_rx):
    rx_pos = rx_list[rx_idx].position.numpy()
    tx_pos = tx_list[target_tx_idx].position.numpy()
    
    # Calculate true AoA/Range (Ground Truth)
    # AoA is measured at RX (AP), pointing toward TX (mobile device)
    true_range = np.linalg.norm(tx_pos - rx_pos)
    direction = tx_pos[:2] - rx_pos[:2]  # Direction FROM RX TO TX
    true_angle = np.arctan2(direction[1], direction[0])
    
    # 1. GENERATE IDEAL CSI
    H_freq_base = np.zeros((NUM_SUBCARRIERS, num_rx_ant), dtype=np.complex128)
    
    for rx_ant in range(num_rx_ant):
        for tx_ant in range(num_tx_ant):
            # Fourier Conversion: CIR to CSI (preserve complex phase!)
            delays = tau_snap[rx_idx, target_tx_idx, :]
            complex_amplitudes = a_snap[rx_idx, rx_ant, target_tx_idx, tx_ant, :]  # Keep complex, not abs()
            for l in range(num_paths):
                H_freq_base[:, rx_ant] += complex_amplitudes[l] * np.exp(-1j * 2*np.pi * delays[l] * subcarriers)
    
    # Transpose CSI to [Rx Antennas, Subcarriers]
    csi_mimo = H_freq_base.T 

    # 2. ADD NON-IDEALITIES (IMPAIRMENT)
    csi_sfo = add_sfo(csi_mimo, subcarrier_indices, sfo_ppm=5)
    csi_pn = add_phase_noise(csi_sfo, pn_std=0.5)

    # Add AWGN (SNR = 20 dB)
    snr_db = 20
    signal_power = np.mean(np.abs(csi_pn)**2)
    noise_power = signal_power * 10**(-snr_db/10)
    noise = np.sqrt(noise_power/2) * (np.random.randn(*csi_pn.shape) + 1j * np.random.randn(*csi_pn.shape))
    final_csi = csi_pn + noise
    
    # 3. APPLY COMPENSATION (SFO + Phase Noise)
    csi_sfo_compensated = compensate_sfo(final_csi, subcarrier_indices)
    csi_compensated = compensate_phase_noise(csi_sfo_compensated)
    avg_csi_final = np.mean(csi_compensated, axis=0)
    
    # 4. DERIVE LOCALIZATION PARAMETERS
    # Use full 2x2 planar array with proper steering vector
    estimated_angle, spectrum = aoa_estimator.estimate_aoa_music(csi_compensated, return_spectrum=True)
    peak_prominence = spectrum.max() / (np.mean(spectrum)+1e-12)
    if peak_prominence < 3:   # Lowered threshold to allow weaker but valid estimates
        estimated_angle = np.nan   # don't use a bad AoA
    
    # First-arrival ranging from CIR (more robust than phase slope in multipath)
    delays = tau_snap[rx_idx, target_tx_idx, :]
    amplitudes = np.mean([a_snap[rx_idx, ant, target_tx_idx, 0, :] for ant in range(num_rx_ant)], axis=0)
    estimated_range = range_estimator.estimate_range_from_cir(delays, amplitudes)

    # Store results - RX positions are the ANCHORS (known AP locations)
    all_estimated_ranges.append(estimated_range)
    all_estimated_angles.append(float(estimated_angle))
    all_anchor_positions.append(rx_pos)  # RX (AP) is the anchor
    
    true_angle_scalar = float(true_angle)
    
    print(f"RX-{rx_idx} (AP) measuring TX-{target_tx_idx}: Range Est={estimated_range:.2f}m (True={true_range:.2f}m)")
    print(f"                                      Angle Est={np.rad2deg(estimated_angle):.1f}° (True={np.rad2deg(true_angle_scalar):.1f}°)")


Localizing TX-0 (Mobile Device) using RX measurements (APs)
True TX-0 Position: [-2.50, 0.50, 0.00]
RX-0 (AP) measuring TX-0: Range Est=3.05m (True=3.05m)
                                      Angle Est=nan° (True=-172.9°)


/var/folders/q1/5dnc4m896ylgx5wy_tycp0bw0000gn/T/ipykernel_45289/1972161919.py:90: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  true_angle_scalar = float(true_angle)


In [54]:
# ============================================================================
# FINAL LOCALIZATION DERIVATION AND RESULTS
# ============================================================================

print("\n" + "=" * 70)
print(f"Performing Hybrid Bilateration to Localize TX-{target_tx_idx}")
print("=" * 70)

# Convert results to NumPy arrays
anchor_pos_array_final = np.array(all_anchor_positions)
ranges_final = np.array(all_estimated_ranges)
angles_final = np.array(all_estimated_angles)

# Debug information
print(f"\nInput Data Summary:")
print(f"  Number of anchors (RX/APs): {len(anchor_pos_array_final)}")
print(f"  Anchor positions shape: {anchor_pos_array_final.shape}")
print(f"  Ranges shape: {ranges_final.shape}")
print(f"  Angles shape: {angles_final.shape}")

# Perform hybrid bilateration
try:
    estimated_position = solver.hybrid_localization(
        anchor_pos_array_final,
        ranges_final,
        angles_final,
        distance_std=0.5, 
        angle_std=np.deg2rad(5)
    )
    
    # Flatten arrays to ensure they're 1D
    estimated_position = np.asarray(estimated_position).flatten()
    true_target_position_flat = np.asarray(true_target_position).flatten()
    
    # Calculate error
    localization_error = np.linalg.norm(estimated_position - true_target_position_flat)
    
    print(f"\n{'='*60}")
    print(f"FINAL LOCALIZATION RESULTS FOR TX-{target_tx_idx}")
    print(f"{'='*60}")
    print(f"True TX Position:      [{true_target_position_flat[0]:.2f}, {true_target_position_flat[1]:.2f}, {true_target_position_flat[2]:.2f}]")
    print(f"Estimated TX Position: [{estimated_position[0]:.2f}, {estimated_position[1]:.2f}, {estimated_position[2]:.2f}]")
    print(f"Localization Error: {localization_error:.3f} meters")
    
    # Per-dimension errors
    error_xyz = estimated_position - true_target_position_flat
    print(f"\nPer-Dimension Errors:")
    print(f"  X-axis error: {error_xyz[0]:+.3f} m")
    print(f"  Y-axis error: {error_xyz[1]:+.3f} m")
    print(f"  Z-axis error: {error_xyz[2]:+.3f} m")
    
    # Individual link statistics
    print(f"\nIndividual Link Performance:")
    for i in range(len(anchor_pos_array_final)):
        anchor_pos = anchor_pos_array_final[i]
        est_range = ranges_final[i]
        est_angle = angles_final[i]
        
        true_range = np.linalg.norm(true_target_position_flat - anchor_pos)
        range_error = abs(float(est_range) - float(true_range))
        
        # Calculate direction with explicit scalar extraction
        dir_y = float(true_target_position_flat[1] - anchor_pos[1])
        dir_x = float(true_target_position_flat[0] - anchor_pos[0])
        true_angle = np.arctan2(dir_y, dir_x)
        angle_error = np.rad2deg(abs(float(est_angle) - float(true_angle)))
        
        print(f"  RX-{i} (AP): Range error = {range_error:.3f}m, Angle error = {angle_error:.2f}°")
    
    print(f"{'='*60}")
    
except Exception as e:
    print(f"\n❌ Localization failed with error:")
    print(f"   {type(e).__name__}: {str(e)}")
    print(f"\nDebug information:")
    print(f"  anchor_pos_array_final:\n{anchor_pos_array_final}")
    print(f"  ranges_final: {ranges_final}")
    print(f"  angles_final: {angles_final}")
    raise


Performing Hybrid Bilateration to Localize TX-0

Input Data Summary:
  Number of anchors (RX/APs): 1
  Anchor positions shape: (1, 3, 1)
  Ranges shape: (1,)
  Angles shape: (1,)

FINAL LOCALIZATION RESULTS FOR TX-0
True TX Position:      [-2.50, 0.50, 0.00]
Estimated TX Position: [-1.19, 0.17, -1.56]
Localization Error: 2.067 meters

Per-Dimension Errors:
  X-axis error: +1.314 m
  Y-axis error: -0.332 m
  Z-axis error: -1.560 m

Individual Link Performance:
  RX-0 (AP): Range error = 2.911m, Angle error = nan°


/var/folders/q1/5dnc4m896ylgx5wy_tycp0bw0000gn/T/ipykernel_45289/1403062533.py:63: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dir_y = float(true_target_position_flat[1] - anchor_pos[1])
/var/folders/q1/5dnc4m896ylgx5wy_tycp0bw0000gn/T/ipykernel_45289/1403062533.py:64: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dir_x = float(true_target_position_flat[0] - anchor_pos[0])


In [55]:
# ============================================================================
# RANGE-ONLY LOCALIZATION (RTT/FTM WITHOUT AoA)
# ============================================================================

print("\n" + "=" * 70)
print(f"Performing Range-Only Localization (RTT/FTM only, no AoA)")
print("=" * 70)

# Simulate RTT-only measurements with timing impairments
# RTT is affected by: clock drift, processing delays, multipath (less than CSI)
rtt_ranges = []

print("\nSimulating RTT-only measurements:")
for i, (rx_pos, true_range) in enumerate(zip(anchor_pos_array_final, ranges_final)):
    # Calculate true range
    true_rtt_range = np.linalg.norm(true_target_position - rx_pos)
    
    # Add RTT-specific impairments
    # 1. Clock drift/offset (systematic error) - typically 10-50 ns
    clock_offset_ns = np.random.normal(0, 20)  # 20 ns std dev
    clock_error_m = (clock_offset_ns * 1e-9) * 3e8 / 2  # Convert to meters (two-way)
    
    # 2. Processing delay variation (random per measurement)
    processing_delay_ns = np.random.uniform(0, 10)  # 0-10 ns
    processing_error_m = (processing_delay_ns * 1e-9) * 3e8 / 2
    
    # 3. Multipath adds positive bias (signal takes longer path)
    multipath_bias_m = np.random.exponential(0.3)  # Exponential distribution, mean 0.3m
    
    # 4. Measurement noise (AWGN on time-of-arrival)
    noise_std_ns = 5  # 5 ns noise standard deviation
    noise_m = np.random.normal(0, noise_std_ns * 1e-9 * 3e8 / 2)
    
    # Total RTT range estimate with impairments
    rtt_range = true_rtt_range + clock_error_m + processing_error_m + multipath_bias_m + noise_m
    rtt_ranges.append(rtt_range)
    
    print(f"  RX-{i}: True={true_rtt_range:.3f}m, RTT Est={rtt_range:.3f}m, Error={rtt_range-true_rtt_range:+.3f}m")

rtt_ranges = np.array(rtt_ranges)

# Use NaN for all angles (no angle information in RTT-only)
angles_nan = np.full_like(angles_final, np.nan)

try:
    estimated_position_rtt_only = solver.hybrid_localization(
        anchor_pos_array_final,
        rtt_ranges,  # Use RTT-specific range estimates
        angles_nan,  # No angle information
        distance_std=0.5, 
        angle_std=np.deg2rad(5)
    )
    
    # Flatten arrays
    estimated_position_rtt_only = np.asarray(estimated_position_rtt_only).flatten()
    
    # Calculate error
    localization_error_rtt_only = np.linalg.norm(estimated_position_rtt_only - true_target_position)
    
    print(f"\n{'='*60}")
    print(f"RTT-ONLY LOCALIZATION RESULTS FOR TX-{target_tx_idx}")
    print(f"{'='*60}")
    print(f"True TX Position:           [{true_target_position[0]:.2f}, {true_target_position[1]:.2f}, {true_target_position[2]:.2f}]")
    print(f"Estimated TX Position:      [{estimated_position_rtt_only[0]:.2f}, {estimated_position_rtt_only[1]:.2f}, {estimated_position_rtt_only[2]:.2f}]")
    print(f"Localization Error:         {localization_error_rtt_only:.3f} meters")
    
    # Per-dimension errors
    error_xyz_rtt = estimated_position_rtt_only - true_target_position
    print(f"\nPer-Dimension Errors:")
    print(f"  X-axis error: {error_xyz_rtt[0]:+.3f} m")
    print(f"  Y-axis error: {error_xyz_rtt[1]:+.3f} m")
    print(f"  Z-axis error: {error_xyz_rtt[2]:+.3f} m")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"\n❌ RTT-only localization failed with error:")
    print(f"   {type(e).__name__}: {str(e)}")
    raise


Performing Range-Only Localization (RTT/FTM only, no AoA)

Simulating RTT-only measurements:
  RX-0: True=5.963m, RTT Est=5.660m, Error=-0.303m

RTT-ONLY LOCALIZATION RESULTS FOR TX-0
True TX Position:           [-2.50, 0.50, 0.00]
Estimated TX Position:      [-1.28, 0.49, -2.56]
Localization Error:         2.840 meters

Per-Dimension Errors:
  X-axis error: +1.220 m
  Y-axis error: -0.014 m
  Z-axis error: -2.564 m


In [56]:
# ============================================================================
# COMPARISON: HYBRID vs RTT-ONLY LOCALIZATION
# ============================================================================

print("\n" + "=" * 70)
print("COMPARISON: HYBRID (CSI Range+Angle) vs RTT-ONLY")
print("=" * 70)

# Calculate improvement
error_reduction = localization_error_rtt_only - localization_error
improvement_percentage = (error_reduction / localization_error_rtt_only) * 100 if localization_error_rtt_only > 0 else 0

print(f"\nLocalization Method Comparison:")
print(f"  {'Method':<30} {'Error (m)':<12} {'X-Error (m)':<12} {'Y-Error (m)':<12} {'Z-Error (m)':<12}")
print(f"  {'-'*78}")
print(f"  {'Hybrid (CSI: Range + Angle)':<30} {localization_error:<12.3f} {error_xyz[0]:<+12.3f} {error_xyz[1]:<+12.3f} {error_xyz[2]:<+12.3f}")
print(f"  {'RTT-Only (Time-of-Flight)':<30} {localization_error_rtt_only:<12.3f} {error_xyz_rtt[0]:<+12.3f} {error_xyz_rtt[1]:<+12.3f} {error_xyz_rtt[2]:<+12.3f}")

print(f"\nPerformance Improvement:")
print(f"  Error Reduction:        {error_reduction:+.3f} meters")
print(f"  Improvement:            {improvement_percentage:+.1f}%")

if localization_error < localization_error_rtt_only:
    print(f"  ✓ Hybrid CSI-based localization performs BETTER (lower error)")
elif localization_error > localization_error_rtt_only:
    print(f"  ⚠ RTT-only localization performs BETTER (lower error)")
else:
    print(f"  = Both methods have equal performance")

# Position comparison
position_difference = np.linalg.norm(estimated_position - estimated_position_rtt_only)
print(f"\nPosition Estimate Difference: {position_difference:.3f} meters")
print(f"  (Distance between hybrid CSI and RTT-only estimates)")

# Compare average range errors
csi_range_errors = [abs(r1 - r2) for r1, r2 in zip(ranges_final, [np.linalg.norm(true_target_position - anchor_pos_array_final[i]) for i in range(len(anchor_pos_array_final))])]
rtt_range_errors = [abs(r1 - r2) for r1, r2 in zip(rtt_ranges, [np.linalg.norm(true_target_position - anchor_pos_array_final[i]) for i in range(len(anchor_pos_array_final))])]

print(f"\nRange Estimation Accuracy:")
print(f"  CSI-based (FTM) avg error:  {np.mean(csi_range_errors):.3f} m")
print(f"  RTT-based avg error:        {np.mean(rtt_range_errors):.3f} m")

print(f"\n{'='*70}")
print("ANALYSIS NOTES:")
print("- Hybrid: Uses CSI phase slope (FTM) for range + MUSIC for AoA")
print("  * Impairments: SFO, phase noise, AWGN affect CSI")
print("- RTT-only: Uses time-of-flight measurements")
print("  * Impairments: Clock drift, processing delay, multipath bias, timing noise")
print("- CSI provides finer-grained phase information for ranging")
print("- AoA from CSI adds directional constraints")
print("- With only 1 RX anchor, both methods may have limited accuracy")
print("- More RX anchors would significantly improve both methods")
print(f"{'='*70}")


COMPARISON: HYBRID (CSI Range+Angle) vs RTT-ONLY

Localization Method Comparison:
  Method                         Error (m)    X-Error (m)  Y-Error (m)  Z-Error (m) 
  ------------------------------------------------------------------------------
  Hybrid (CSI: Range + Angle)    2.067        +1.314       -0.332       -1.560      
  RTT-Only (Time-of-Flight)      2.840        +1.220       -0.014       -2.564      

Performance Improvement:
  Error Reduction:        +0.773 meters
  Improvement:            +27.2%
  ✓ Hybrid CSI-based localization performs BETTER (lower error)

Position Estimate Difference: 1.057 meters
  (Distance between hybrid CSI and RTT-only estimates)

Range Estimation Accuracy:
  CSI-based (FTM) avg error:  2.911 m
  RTT-based avg error:        0.303 m

ANALYSIS NOTES:
- Hybrid: Uses CSI phase slope (FTM) for range + MUSIC for AoA
  * Impairments: SFO, phase noise, AWGN affect CSI
- RTT-only: Uses time-of-flight measurements
  * Impairments: Clock drift, processin